# DFST Arithmetic

Compare training budgets and warmup durations for binary and decimal addition and multiplication.

Install the package with `pip install -e ".[analysis]"` from the repository root. This notebook reads the final evaluations selected in `paper/figures.yaml`. The notebook uses saved results without training or checkpoint evaluation.

Every selected run requires a final evaluation. Zero denotes a measured result; `≥` marks a tested ceiling. The notebook saves task figures under `analysis_outputs/paper_figures/DFST/<task>/` and combined tables and figures under `DFST/arithmetic/`. `DEL_ANALYSIS_DIR` selects another figure root and preserves these subfolders.


In [ ]:
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path(os.environ["DEL_REPO_ROOT"]) if os.environ.get("DEL_REPO_ROOT") else next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/differentiable_exact_learning").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from differentiable_exact_learning.analysis.release import figure_group, final_results_table
from differentiable_exact_learning.analysis.figure_data import panel_data
from differentiable_exact_learning.analysis.paper_heatmaps import plot_heatmap_group, save_figure, figure_output_dir
from differentiable_exact_learning.analysis.plot_style import apply_paper_style
apply_paper_style()
OUTPUT_DIR = figure_output_dir(ROOT, 'DFST')
SUMMARY_DIR = figure_output_dir(ROOT, 'DFST', "arithmetic")
def export(fig, name):
    paths = save_figure(fig, OUTPUT_DIR / name)
    display(fig)
    plt.close(fig)
    return paths

In [ ]:
TASKS = ("add2", "mult2", "add10", "mult10")
records = {task: figure_group(ROOT, f"DFST_{task}") for task in TASKS}
results = final_results_table([row for rows in records.values() for row in rows])
display(results)

## Warmup and Training Budgets

Each cell represents a final checkpoint at the training budget shown on the vertical axis. Addition and multiplication use separate color scales.

In [ ]:
for metric in ("ulg", "rlg"):
    panels = []
    for task in TASKS:
        rows = records[task]
        xs = sorted({r["config"]["training"]["warmup_steps"] for r in rows})
        ys = sorted({r["config"]["training"]["steps"] for r in rows})
        panel = panel_data(rows, xs, ys, metric=metric, title=task)
        panel["cap"] = 5000 if task.startswith("add") else 1000
        panels.append(panel)
        export(plot_heatmap_group([panel], metric.upper(), ncols=1), f"{task}/dfst_{task}_{metric}")
    export(plot_heatmap_group(panels, metric.upper(), ncols=4), f"arithmetic/dfst_{metric}_combined")
results.to_csv(SUMMARY_DIR / "dfst_arithmetic.csv", index=False)